In [1]:
import torch
from test_attn_order_eval import AttnOrderEval   # signal-agnostic ranking eval over a [T, L] per-step score + order


class ConfidenceOrderEval(AttnOrderEval):

    def __init__(self, confidence, order):                          # confidence [T, L] (already preprocessed), order [T] long
        assert confidence.dim() == 2, "confidence.dim(): {} == 2 false".format(confidence.dim())

        super().__init__(confidence, order)                         # reuse geometry + recall_at_h / pr_auc / ndcg_at_h
    # end

    def get_confidence(self):
        return self.attn   # the base stores the ranking score here (generic [T, L] score slot)
    # end


In [ ]:
# if __name__ == "__main__":
#     torch.manual_seed(0)
#     T = L = 64
#     order = torch.randperm(L)

#     step_of = torch.full((L,), -1, dtype=torch.long)
#     step_of[order] = torch.arange(T)
#     gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

#     conf_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))   # higher conf = sooner
#     conf_random = torch.rand(T, L, dtype=torch.float64)

#     def summ(x):
#         return "{:.3f} (n={})".format(x.nanmean().item(), int((~x.isnan()).sum()))
#     # end

#     for name, conf in [("perfect", conf_perfect), ("random ", conf_random)]:
#         ev = ConfidenceOrderEval(conf, order)
#         print("{}  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
#             name, summ(ev.recall_at_h(5)), summ(ev.pr_auc(5)), summ(ev.ndcg_at_h(10))))
#     # end

In [3]:
if __name__ == "__main__":
    torch.manual_seed(0)
    T = L = 64
    
    folder_base = 'stats/0'
    pos_base = 32

    unmask = torch.load(f'{folder_base}/unmask_{pos_base}_{pos_base + L}.pt')
    unmask = unmask.squeeze(-1) - pos_base
    order = unmask

    step_of = torch.full((L,), -1, dtype=torch.long)
    step_of[order] = torch.arange(T)
    gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

    conf_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))   # higher conf = sooner
    conf_random = torch.rand(T, L, dtype=torch.float64)
    conf = torch.load(f'{folder_base}/conf_{pos_base}_{pos_base + L}.pt')

    def summ(x):
        return "{:.3f} (n={})".format(x.nanmean().item(), int((~x.isnan()).sum()))
    # end

    for name, conf in [("perfect", conf_perfect), ("random ", conf_random), ("conf", conf)]:
        ev = ConfidenceOrderEval(conf, order)
        print("{}  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
            name, summ(ev.recall_at_h(5)), summ(ev.pr_auc(5)), summ(ev.ndcg_at_h(10))))
    # end


perfect  recall@5=1.000 (n=59)  pr_auc@5=1.000 (n=58)  ndcg@10=1.000 (n=62)
random   recall@5=0.275 (n=59)  pr_auc@5=0.319 (n=58)  ndcg@10=0.412 (n=62)
conf  recall@5=0.766 (n=59)  pr_auc@5=0.829 (n=58)  ndcg@10=0.869 (n=62)
